# Acacias Sound Monitor: End-to-End Visual Demo
## Phase 5: Stakeholder Presentation

This notebook serves as an interactive demonstration of our Sound Event Detection (SED) system.

**Objectives:**
1. Pick a raw 10-second urban soundscape.
2. Listen to the raw audio.
3. Compute the Mel Spectrogram 
4. Perform live inference using our trained `SB_CNN_SED` PyTorch model.
5. Plot the model's predicted event probabilities synchronized over time to show exactly when sounds occur.

In [ ]:
import sys
from pathlib import Path
import torch
import torchaudio
import matplotlib.pyplot as plt
import librosa.display
import numpy as np
import pandas as pd
from IPython.display import Audio, display

# Ensure src modules are discoverable
sys.path.append(str(Path.cwd().parent / "src"))

from sbcnn_sed.data.features import MelSpectrogramExtractor
from sbcnn_sed.utils.scaler import MinMaxScaler
from sbcnn_sed.model.models import SBCNNSed
from sbcnn_sed.data.dataset import URBAN_SED_CLASSES

# Configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'mps' if torch.backends.mps.is_available() else 'cpu')
print(f"Operational device: {device}")

# Paths to our raw data and trained artifacts
# We select a file from the unseen TEST fold for absolute fairness
TEST_WAV_PATH = "../data/raw/URBAN-SED_v2.0.0/audio/test/soundscape_test_bimodal1.wav"
TEST_ANNOTATION_PATH = "../data/raw/URBAN-SED_v2.0.0/annotations/test/soundscape_test_bimodal1.txt"
SCALER_PATH = "../data/processed/URBAN-SED_v2.0.0/scaler.pt"
MODEL_WEIGHTS = "../models/checkpoints/best_sed_model.pth"

print(f"Testing File: {Path(TEST_WAV_PATH).name}")

## 1. Listen to the Audio Payload

Before looking at data, let's establish intuition. What does the raw waveform sound like?

In [ ]:
waveform, sample_rate = torchaudio.load(TEST_WAV_PATH)
time_axis = np.linspace(0, waveform.shape[1] / sample_rate, num=waveform.shape[1])

plt.figure(figsize=(12, 2))
plt.plot(time_axis, waveform[0].numpy(), color="navy", alpha=0.7)
plt.title(f"Raw Waveform - {Path(TEST_WAV_PATH).name}")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")
plt.tight_layout()
plt.show()

display(Audio(TEST_WAV_PATH))

## 2. Load the Ground Truth (What sounds are actually there?)

For this demo, we will look up the `annotations/` file that corresponds to this soundscape. This text file tells us the exact start and stop times of the simulated sound events.

In [ ]:
# Read ground truth annotations
gt_df = pd.read_csv(TEST_ANNOTATION_PATH, sep="\t", header=None, names=["onset", "offset", "event_label"])

print("Ground Truth Annotations for this 10-second clip:")
display(gt_df)

In [ ]:
import matplotlib.patches as patches

# Visualize the ground truth timeline
fig, ax = plt.subplots(figsize=(12, 3))
ax.set_title(f"Ground Truth Event Timeline - {Path(TEST_ANNOTATION_PATH).name}")
ax.set_xlabel("Time (seconds)")
ax.set_ylabel("Event Categories")
ax.set_xlim(0, 10)  # Urban-SED files are 10s long

# Get unique labels to position them on the y-axis
unique_labels = sorted(gt_df['event_label'].unique())
y_positions = {label: i for i, label in enumerate(unique_labels)}
ax.set_yticks(range(len(unique_labels)))
ax.set_yticklabels(unique_labels)

# Plot each event as a horizontal bar
colors = plt.cm.tab10(np.linspace(0, 1, 10))
for _, row in gt_df.iterrows():
    label = row['event_label']
    y_pos = y_positions[label]
    duration = row['offset'] - row['onset']
    # Add a rectangle patch for the event
    rect = patches.Rectangle((row['onset'], y_pos - 0.3), duration, 0.6, 
                             linewidth=1, edgecolor='black', facecolor=colors[URBAN_SED_CLASSES.index(label)], alpha=0.7)
    ax.add_patch(rect)

ax.grid(True, axis='x', linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

## 3. Feature Extraction (Spectrogram)

The neural network does not process the blue waveform directly. It requires a 2D time-frequency representation called a **Mel Spectrogram**. 

We now extract it exactly as our pipeline did during training, and apply the `MinMaxScaler` that was fitted exclusively on the training set.

In [ ]:
# Initialize exactly matching extractor
extractor = MelSpectrogramExtractor(
    sample_rate=22050, n_fft=1024, win_length=1024, hop_length=680, 
    n_mels=64, sequence_time=1.0, sequence_hop_time=0.5
)

# Extract raw sequences: shape will be [19, 32, 64] -> 19 sequences of 1s length
raw_sequences = extractor.extract(TEST_WAV_PATH).to(torch.float32)

# Apply identical training scaler
scaler = MinMaxScaler()
scaler.load(SCALER_PATH)
scaled_sequences = scaler.transform(raw_sequences)

print(f"Extracted feature shape: {scaled_sequences.shape} (Sequences, Time Frames, Mel Bands)")

In [ ]:
spectrogram_continuous = librosa.feature.melspectrogram(
    y=waveform[0].numpy(), 
    sr=sample_rate, 
    n_fft=extractor.n_fft, 
    hop_length=extractor.hop_length, 
    n_mels=extractor.n_mels
)
spectrogram_db = librosa.power_to_db(spectrogram_continuous, ref=np.max)

fig, ax1 = plt.subplots(figsize=(14, 4))
librosa.display.specshow(spectrogram_db, sr=sample_rate, 
                         hop_length=extractor.hop_length, x_axis='time', y_axis='mel', 
                         ax=ax1, cmap='magma')
ax1.set_title("Input Spectrogram with Ground Truth (White Bounds)")

# Draw ground truth bounding boxes
for _, row in gt_df.iterrows():
    ax1.axvspan(row['onset'], row['offset'], color='white', alpha=0.3, ymin=0.1, ymax=0.9)
    ax1.text(row['onset'] + 0.1, 8000, row['event_label'], color='white', fontsize=10, weight='bold')

ax1.set_xlim(0, 10)
plt.tight_layout()
plt.show()

## 4. Live Model Inference

We load the `SB_CNN_SED` network weights trained in Phase 3. 
We push our 19 `1-second` sequences through the network. The Output will be 19 rows representing the probabilities (0.0 to 1.0) of each of the 10 sound classes at every half-second step.

In [ ]:
NEW_FILE_PATH = "../data/raw/URBAN-SED_v2.0.0/audio/test/soundscape_test_bimodal3.wav"

# 1. Load the new raw audio (needed for the spectrogram plot in Section 5)
waveform, sample_rate = torchaudio.load(NEW_FILE_PATH)

# 2. Extract the Mel Spectrogram features
raw_sequences = extractor.extract(NEW_FILE_PATH).to(torch.float32)

# 3. Apply the scaler so the model understands the input
scaled_sequences = scaler.transform(raw_sequences)

# Update the spectrogram graphic for Section 5
spectrogram_continuous = librosa.feature.melspectrogram(
    y=waveform[0].numpy(), 
    sr=sample_rate, 
    n_fft=extractor.n_fft, 
    hop_length=extractor.hop_length, 
    n_mels=extractor.n_mels
)
spectrogram_db = librosa.power_to_db(spectrogram_continuous, ref=np.max)

In [ ]:
# Load the Model
model = SBCNNSed(num_classes=10).to(device)

try:
    checkpoint = torch.load(MODEL_WEIGHTS, map_location=device)
    model.load_state_dict(checkpoint["model_state_dict"])
    print("Model weights successfully loaded!")
except FileNotFoundError:
    print(f"Warning: Model weights not found at {MODEL_WEIGHTS}. Ensure you have trained the model first.")

model.eval()

# Prepare tensor layout for PyTorch Conv2D [Batch, Channels, Time, Mels]
num_sequences, height, width = scaled_sequences.shape
inference_tensor = scaled_sequences.view(num_sequences, 1, height, width).to(device)

# Predict probabilities
with torch.no_grad():
    # model.predict gives us the activated sigmoid logits (0 to 1 probabilities)
    probabilities = model.predict(inference_tensor).cpu().numpy()

print(f"Output probability shape: {probabilities.shape} (Time Steps, Sound Classes)")

## 5. Visualizing the Engine's Decisions

We now plot the model's confidence scores over time to visualize exactly when the model "hears" something. This curve will stretch across the same 10-second timeline.

In [ ]:
# Calculate time axis mapping for our 19 prediction steps
time_steps = np.arange(num_sequences) * extractor.sequence_hop_time + (extractor.sequence_time/2)

fig, (ax1, ax2) = plt.subplots(nrows=2, ncols=1, figsize=(14, 8), sharex=True, gridspec_kw={'height_ratios': [1, 2]})

# --- Plot 1: Spectrogram ---
librosa.display.specshow(spectrogram_db, sr=sample_rate, 
                         hop_length=extractor.hop_length, x_axis='time', y_axis='mel', 
                         ax=ax1, cmap='magma')
ax1.set_title("Input Spectrogram (Model's Perspective)")

# --- Plot 2: Model Prediction Curves ---
colors = plt.cm.tab10(np.linspace(0, 1, 10))

# Only plot classes that the model is confident about or exist in ground truth to avoid clutter
active_classes_indices = np.where(np.max(probabilities, axis=0) > 0.4)[0]

for idx in active_classes_indices:
    ax2.plot(time_steps, probabilities[:, idx], label=URBAN_SED_CLASSES[idx], 
             color=colors[idx], linewidth=3, marker='o')
    # Fill under curve for visibility
    ax2.fill_between(time_steps, probabilities[:, idx], alpha=0.1, color=colors[idx])

ax2.axhline(0.5, color='gray', linestyle='--', alpha=0.7, label='Threshold (0.5)')
ax2.set_title("Deep Learning Model Deductions (Live Probabilities)")
ax2.set_xlabel("Time (seconds)")
ax2.set_ylabel("Confidence Probability")
ax2.set_xlim(0, 10) # Enforce strict 10 second timeline
ax2.set_ylim(-0.05, 1.05)
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()
display(Audio(NEW_FILE_PATH))

In [1]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent / "src"))

from sbcnn_sed.pipeline.inference import SoundEventDetector

# 1. Initialize the detector with your config
detector = SoundEventDetector("../configs/inference.yaml")

# 2. Run prediction on a test audio file
predictions = detector.predict("../data/raw/URBAN-SED_v2.0.0/audio/test/soundscape_test_bimodal0.wav")

# 3. Print the resulting JSON
import json
print(json.dumps(predictions, indent=2))

/home/julia/juli/projects/marso/acacias-sound-monitor/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


[
  {
    "event": "gun_shot",
    "start": 2.5,
    "end": 4.5,
    "confidence": 0.9622725248336792
  },
  {
    "event": "air_conditioner",
    "start": 4.0,
    "end": 7.5,
    "confidence": 0.7624034285545349
  },
  {
    "event": "engine_idling",
    "start": 4.5,
    "end": 6.0,
    "confidence": 0.5968477725982666
  }
]
